# Neonatal LUS Edge AI Triage — Full Pipeline Notebook

**Repository:** [jeevikabhardwaj/neonatal-edge-triage](https://github.com/jeevikabhardwaj/neonatal-edge-triage)

## Overview
Two-phase AI triage system for Neonatal Lung Ultrasound (LUS):

| Phase | Task | Model | Data | Result |
|-------|------|-------|------|--------|
| Phase 1 | Binary: Normal vs Abnormal | MobileNetV3-Small | 904 real frames (covid19_ultrasound) | ROC-AUC = **0.9989** |
| Phase 2 | 3-class: Normal / Moderate / High-Risk | MobileNetV3 + Clinical MLP | 500 synthetic neonatal patients | ROC-AUC = **0.8489** |

Both models are exported as TorchScript for edge deployment (~4 MB each).

---
## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Data Pipeline — Download & Extract](#2-data-pipeline)
3. [Build Manifest (Stratified Patient-Level Split)](#3-build-manifest)
4. [Phase 1 — Binary Classifier Training](#4-phase-1-training)
5. [Phase 1 — Evaluation & ROC-AUC](#5-phase-1-evaluation)
6. [Synthetic Neonatal Data Generation](#6-synthetic-data)
7. [Phase 2 — Multimodal Training](#7-phase-2-training)
8. [Phase 2 — Evaluation (3-Class)](#8-phase-2-evaluation)
9. [TorchScript Export for Edge Deployment](#9-export)
10. [Demo Inference (Gradio App)](#10-demo)


## 1. Environment Setup


In [ ]:
# Install dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu -q
!pip install scikit-learn matplotlib seaborn pandas pillow tqdm gradio python-docx openpyxl gdown -q

import sys, os
print(f'Python {sys.version}')

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
from pathlib import Path

# Project paths
PROJECT_ROOT = Path('.').resolve()
DATA_DIR     = PROJECT_ROOT / 'data'
RAW_DIR      = DATA_DIR / 'raw'
NORMAL_DIR   = RAW_DIR / 'normal'
ABNORMAL_DIR = RAW_DIR / 'abnormal'
MANIFEST_CSV = DATA_DIR / 'manifest.csv'
MODELS_DIR   = PROJECT_ROOT / 'models'

for d in [NORMAL_DIR, ABNORMAL_DIR, MODELS_DIR / 'phase1', MODELS_DIR / 'phase2']:
    d.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('All directories created.')

## 2. Data Pipeline — Download & Extract

We use `jannisborn/covid19_ultrasound` (GitHub LFS) as our real LUS data source.
Label mapping: `Reg_*` → normal, everything else (`Cov_*`, `Pneu_*`, `Vir_*`) → abnormal.


In [ ]:
# Check if data already extracted
normal_count   = len(list(NORMAL_DIR.glob('*.png'))) + len(list(NORMAL_DIR.glob('*.jpg')))
abnormal_count = len(list(ABNORMAL_DIR.glob('*.png'))) + len(list(ABNORMAL_DIR.glob('*.jpg')))
print(f'Normal frames:   {normal_count}')
print(f'Abnormal frames: {abnormal_count}')
print(f'Total:           {normal_count + abnormal_count}')

In [ ]:
# If data not yet extracted, run the extraction script
# This downloads covid19_ultrasound via git clone + LFS and extracts video frames
# Expected output: ~904 frames (252 normal, 652 abnormal)

if normal_count + abnormal_count < 100:
    print('Running frame extraction...')
    !python scripts/extract_openpocus_frames.py
else:
    print(f'Data already extracted: {normal_count} normal, {abnormal_count} abnormal')

## 3. Build Manifest (Stratified Patient-Level Split)

Key design: **stratified patient-level split** — patients split 70/15/15 independently per class.
This prevents frame leakage AND ensures balanced test sets (critical for unbiased ROC-AUC).

**Why this matters:** Without stratification, test set had 4.4:1 abnormal:normal ratio (101 vs 23 samples), causing ROC-AUC to drop to 0.394. With stratified split: 37.4% normal in test → ROC-AUC = 0.9989.


In [ ]:
import numpy as np
import pandas as pd

def assign_splits_stratified(df, train_frac=0.70, val_frac=0.15, seed=42):
    """Split patients per class independently to ensure balanced splits."""
    rng = np.random.default_rng(seed)
    patient_label = df.groupby('patient_id')['label'].agg(lambda x: int(x.mode()[0]))
    split_map = {}
    for label_val in sorted(patient_label.unique()):
        patients = sorted(patient_label[patient_label == label_val].index.tolist())
        rng.shuffle(patients)
        n = len(patients)
        n_train = max(1, int(n * train_frac))
        n_val   = max(1, int(n * val_frac))
        if n_train + n_val >= n:
            n_val = max(1, n - n_train - 1)
        for i, pid in enumerate(patients):
            if i < n_train:              split_map[pid] = 'train'
            elif i < n_train + n_val:    split_map[pid] = 'validation'
            else:                        split_map[pid] = 'test'
    return split_map

print('Stratified split function defined.')

In [ ]:
# Build manifest or load existing
if MANIFEST_CSV.exists():
    df = pd.read_csv(MANIFEST_CSV)
    print(f'Loaded existing manifest: {len(df)} frames')
else:
    print('Building manifest...')
    !python scripts/build_manifest.py
    df = pd.read_csv(MANIFEST_CSV)
    print(f'Built manifest: {len(df)} frames')

# Summary
print('\nSplit distribution:')
print(df.groupby(['split', 'label_name']).size().unstack(fill_value=0))

print('\nTest set class balance:')
test = df[df['split'] == 'test']
n_norm = (test['label'] == 0).sum()
n_abn  = (test['label'] == 1).sum()
print(f'  Normal:   {n_norm} ({n_norm/len(test)*100:.1f}%)')
print(f'  Abnormal: {n_abn} ({n_abn/len(test)*100:.1f}%)')

## 4. Phase 1 — Binary Classifier Training

**Architecture:** MobileNetV3-Small (pretrained ImageNet) with custom 2-class head.
- ~1.5M parameters, ~4 MB on disk
- Weighted CrossEntropyLoss for class imbalance
- AdamW optimizer, CosineAnnealing LR schedule
- Best checkpoint saved on validation ROC-AUC


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class LUSDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['frame_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])


def build_phase1_model(num_classes=2):
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model


# Transforms
TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Phase 1 model and dataset classes defined.')
model = build_phase1_model()
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,} (~{total_params*4/1e6:.1f} MB)')

In [ ]:
# Check if Phase 1 checkpoint already exists
P1_CKPT = MODELS_DIR / 'phase1' / 'checkpoints' / 'phase1_best.pth'

if P1_CKPT.exists():
    print(f'Phase 1 checkpoint exists: {P1_CKPT}')
    print('Skip training — loading existing checkpoint.')
else:
    print('No checkpoint found. Running Phase 1 training...')
    print('This takes ~10-20 min on CPU.')
    !python src/training/train_phase1.py --config config/phase1_config.yaml

## 5. Phase 1 — Evaluation & ROC-AUC


In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

P1_REPORT = MODELS_DIR / 'phase1' / 'reports' / 'phase1_eval_report.json'

if P1_REPORT.exists():
    with open(P1_REPORT) as f:
        p1 = json.load(f)
    print('=== Phase 1 Results ===')
    print(f"Accuracy:          {p1.get('accuracy', 'N/A'):.4f}")
    print(f"ROC-AUC:           {p1.get('roc_auc', 'N/A'):.4f}")
    print(f"F1 (abnormal):     {p1.get('f1_abnormal', 'N/A'):.4f}")
    print(f"Recall (normal):   {p1.get('recall_normal', 'N/A'):.4f}")
    print(f"Recall (abnormal): {p1.get('recall_abnormal', 'N/A'):.4f}")
else:
    print('Running Phase 1 evaluation...')
    !python src/evaluation/evaluate_phase1.py

In [ ]:
# Display ROC curve
P1_ROC = MODELS_DIR / 'phase1' / 'reports' / 'roc_curve.png'
P1_CM  = MODELS_DIR / 'phase1' / 'reports' / 'confusion_matrix.png'

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, path, title in zip(axes, [P1_ROC, P1_CM], ['ROC Curve', 'Confusion Matrix']):
    if Path(path).exists():
        ax.imshow(mpimg.imread(str(path)))
        ax.set_title(f'Phase 1 — {title}', fontsize=13)
    else:
        ax.text(0.5, 0.5, f'{title}\nnot yet generated', ha='center', va='center')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Synthetic Neonatal Data Generation

**Why synthetic?** Real neonatal LUS data is scarce and not publicly available.

- **Images:** 904 synthetic neonatal frames derived from the 904 real adult LUS frames (grayscale + neonatal-specific augmentation)
- **Clinical data:** 500 synthetic patients with 8 clinical features sampled from published neonatal RDS distributions:
  - gestational_age_weeks, birth_weight_kg, spo2_percent, fio2_fraction
  - respiratory_rate, chest_retractions, grunting, nasal_flaring
- **Labels:** class_0 (normal), class_1 (moderate), class_2 (high-risk)


In [ ]:
# Check if synthetic data exists
SYNTH_IMG_DIR = DATA_DIR / 'synthetic' / 'neonatal_images'
SYNTH_CLIN    = DATA_DIR / 'synthetic' / 'clinical_data.csv'

synth_img_count = len(list(SYNTH_IMG_DIR.rglob('*.png'))) if SYNTH_IMG_DIR.exists() else 0
print(f'Synthetic images: {synth_img_count}')
print(f'Clinical CSV exists: {SYNTH_CLIN.exists()}')

if synth_img_count == 0:
    print('Generating synthetic neonatal images...')
    !python scripts/generate_neonatal_images.py

if not SYNTH_CLIN.exists():
    print('Generating synthetic clinical data...')
    !python scripts/generate_clinical_data.py

In [ ]:
# Inspect clinical data distribution
if SYNTH_CLIN.exists():
    clin = pd.read_csv(SYNTH_CLIN)
    print(f'Clinical dataset: {clin.shape}')
    print('\nClass distribution:')
    print(clin['label'].value_counts())
    print('\nFeature statistics:')
    display_cols = ['gestational_age_weeks', 'birth_weight_kg', 'spo2_percent',
                    'fio2_fraction', 'respiratory_rate']
    print(clin[display_cols].describe().round(2))

## 7. Phase 2 — Multimodal Training

**Architecture:** Dual-stream fusion
- **Image stream:** MobileNetV3-Small backbone → 576-dim feature vector
- **Clinical stream:** MLP (8 → 32 dim)
- **Fusion:** Concat → Linear → 3-class output

**Training details:**
- Weighted CrossEntropyLoss (inverse class frequency)
- `drop_last=True` on train DataLoader (prevents BatchNorm1d crash on 1-sample batch)
- Best checkpoint saved on validation balanced accuracy


In [ ]:
class ClinicalMLP(nn.Module):
    def __init__(self, n_features=8, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
    def forward(self, x): return self.net(x)


class MultimodalTriage(nn.Module):
    def __init__(self, n_features=8, n_classes=3):
        super().__init__()
        backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        # Remove final classifier; keep features (576-dim)
        self.image_encoder = nn.Sequential(*list(backbone.children())[:-1], nn.Flatten())
        img_dim = 576
        self.clinical_encoder = ClinicalMLP(n_features, hidden=32)
        self.classifier = nn.Sequential(
            nn.Linear(img_dim + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, image, clinical):
        img_feat  = self.image_encoder(image)
        clin_feat = self.clinical_encoder(clinical)
        fused     = torch.cat([img_feat, clin_feat], dim=1)
        return self.classifier(fused)


model_p2 = MultimodalTriage()
total_p2 = sum(p.numel() for p in model_p2.parameters())
print(f'Phase 2 parameters: {total_p2:,} (~{total_p2*4/1e6:.1f} MB)')

In [ ]:
# Check / run Phase 2 training
P2_CKPT = MODELS_DIR / 'phase2' / 'checkpoints' / 'phase2_best.pth'

if P2_CKPT.exists():
    print(f'Phase 2 checkpoint exists: {P2_CKPT}')
    ckpt = torch.load(P2_CKPT, map_location='cpu')
    print(f"Best val balanced acc: {ckpt.get('best_val_balanced_acc', 'N/A')}")
    print(f"Epochs trained: {ckpt.get('epoch', 'N/A')}")
else:
    print('Running Phase 2 training (takes ~15-30 min on CPU)...')
    !python src/training/train_phase2.py --config config/phase2_config.yaml

## 8. Phase 2 — Evaluation (3-Class)


In [ ]:
P2_REPORT = MODELS_DIR / 'phase2' / 'reports' / 'phase2_eval_report.json'

if P2_REPORT.exists():
    with open(P2_REPORT) as f:
        p2 = json.load(f)
    print('=== Phase 2 Results ===')
    print(f"Accuracy:          {p2.get('accuracy', 'N/A'):.4f}")
    print(f"Balanced Accuracy: {p2.get('balanced_accuracy', 'N/A'):.4f}")
    print(f"Macro F1:          {p2.get('macro_f1', 'N/A'):.4f}")
    print(f"ROC-AUC (OvR):     {p2.get('roc_auc_ovr', 'N/A'):.4f}")
    if 'per_class_f1' in p2:
        print('\nPer-class F1:')
        for cls, f1 in p2['per_class_f1'].items():
            print(f'  {cls}: {f1:.4f}')
else:
    print('Running Phase 2 evaluation...')
    !python src/evaluation/evaluate_phase2.py

In [ ]:
# Display Phase 2 evaluation plots
P2_CM  = MODELS_DIR / 'phase2' / 'reports' / 'confusion_matrix.png'
P2_ROC = MODELS_DIR / 'phase2' / 'reports' / 'roc_curves.png'

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, path, title in zip(axes, [P2_CM, P2_ROC], ['Confusion Matrix (3-class)', 'ROC Curves (OvR)']):
    if Path(path).exists():
        ax.imshow(mpimg.imread(str(path)))
        ax.set_title(f'Phase 2 — {title}', fontsize=13)
    else:
        ax.text(0.5, 0.5, f'{title}\nnot yet generated', ha='center', va='center')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. TorchScript Export for Edge Deployment

Both models traced to TorchScript for deployment on edge devices (Raspberry Pi, Jetson Nano, etc.).
Sizes: Phase 1 ≈ 4.07 MB, Phase 2 ≈ 4.25 MB.


In [ ]:
def export_phase1(checkpoint_path, output_path):
    model = build_phase1_model()
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    dummy = torch.zeros(1, 3, 224, 224)
    traced = torch.jit.trace(model, dummy)
    traced.save(str(output_path))
    size_mb = Path(output_path).stat().st_size / 1e6
    print(f'Phase 1 exported: {output_path} ({size_mb:.2f} MB)')


def export_phase2(checkpoint_path, output_path, n_features=8):
    model = MultimodalTriage(n_features=n_features)
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    dummy_img  = torch.zeros(1, 3, 224, 224)
    dummy_clin = torch.zeros(1, n_features)
    traced = torch.jit.trace(model, (dummy_img, dummy_clin))
    traced.save(str(output_path))
    size_mb = Path(output_path).stat().st_size / 1e6
    print(f'Phase 2 exported: {output_path} ({size_mb:.2f} MB)')


P1_TRACED = MODELS_DIR / 'phase1' / 'phase1_traced.pt'
P2_TRACED = MODELS_DIR / 'phase2' / 'phase2_traced.pt'

if P1_CKPT.exists() and not P1_TRACED.exists():
    export_phase1(P1_CKPT, P1_TRACED)
elif P1_TRACED.exists():
    print(f'Phase 1 traced model exists: {P1_TRACED.stat().st_size/1e6:.2f} MB')

if P2_CKPT.exists() and not P2_TRACED.exists():
    export_phase2(P2_CKPT, P2_TRACED)
elif P2_TRACED.exists():
    print(f'Phase 2 traced model exists: {P2_TRACED.stat().st_size/1e6:.2f} MB')

In [ ]:
# Verify TorchScript models load and run
if P1_TRACED.exists():
    m1 = torch.jit.load(str(P1_TRACED))
    m1.eval()
    out = m1(torch.zeros(1, 3, 224, 224))
    print(f'Phase 1 TorchScript output shape: {out.shape}  ✓')

if P2_TRACED.exists():
    m2 = torch.jit.load(str(P2_TRACED))
    m2.eval()
    out = m2(torch.zeros(1, 3, 224, 224), torch.zeros(1, 8))
    print(f'Phase 2 TorchScript output shape: {out.shape}  ✓')

## 10. Demo Inference (Gradio App)

Run the interactive Gradio demo with two tabs:
- **Tab 1:** Phase 1 binary triage (upload any LUS image → Normal / Abnormal + confidence)
- **Tab 2:** Phase 2 multimodal triage (image + 8 clinical sliders → 3-class risk score)


In [ ]:
# Quick inference demo without Gradio UI
import torch.nn.functional as F
from PIL import Image

CLASS_NAMES_P1 = ['Normal', 'Abnormal']
CLASS_NAMES_P2 = ['Normal', 'Moderate Risk', 'High Risk']

def predict_phase1(image_path):
    """Run Phase 1 binary prediction on a single image."""
    if not P1_TRACED.exists():
        return 'Phase 1 model not exported yet.'
    model = torch.jit.load(str(P1_TRACED))
    model.eval()
    img = Image.open(image_path).convert('RGB')
    img = VAL_TRANSFORM(img).unsqueeze(0)
    with torch.no_grad():
        logits = model(img)
        probs  = F.softmax(logits, dim=1)[0]
    pred = CLASS_NAMES_P1[probs.argmax().item()]
    conf = probs.max().item()
    return f'Prediction: {pred} (confidence: {conf:.1%})'


def predict_phase2(image_path, clinical_values: list):
    """Run Phase 2 multimodal prediction."""
    if not P2_TRACED.exists():
        return 'Phase 2 model not exported yet.'
    ckpt = torch.load(P2_CKPT, map_location='cpu')
    mu  = torch.tensor([ckpt['clinical_mu'][k] for k in ckpt['clinical_mu']])
    std = torch.tensor([ckpt['clinical_std'][k] for k in ckpt['clinical_std']])
    model = torch.jit.load(str(P2_TRACED))
    model.eval()
    img = Image.open(image_path).convert('RGB')
    img = VAL_TRANSFORM(img).unsqueeze(0)
    clin = (torch.tensor(clinical_values, dtype=torch.float32) - mu) / std
    clin = clin.unsqueeze(0)
    with torch.no_grad():
        logits = model(img, clin)
        probs  = F.softmax(logits, dim=1)[0]
    pred = CLASS_NAMES_P2[probs.argmax().item()]
    conf = probs.max().item()
    return f'Prediction: {pred} (confidence: {conf:.1%})'


print('Inference functions defined.')
print('To launch Gradio UI: !python app.py')

In [ ]:
# Launch Gradio demo (will open in browser)
# Uncomment the line below to run:
# !python app.py
print('Run:  python app.py')
print('Then open http://localhost:7860 in your browser.')

## Results Summary

| Metric | Phase 1 (Binary) | Phase 2 (3-class) |
|--------|-----------------|-------------------|
| ROC-AUC | **0.9989** | **0.8489** (OvR) |
| Accuracy | 0.8757 | 0.80 |
| Balanced Accuracy | 0.9076 | 0.7487 |
| Macro F1 | 0.88 | 0.75 |
| High-Risk F1 | — | **0.96** |
| Model Size | 4.07 MB | 4.25 MB |

**Key design decisions:**
1. **Stratified patient-level split** — critical for unbiased ROC-AUC (fixed 0.394 → 0.9989)
2. **Weighted loss** — handles class imbalance without resampling
3. **TorchScript export** — enables edge deployment without Python runtime
4. **High-Risk class F1 = 0.96** — most important class (needs immediate intervention) detected near-perfectly

---
*Data: `jannisborn/covid19_ultrasound` (904 real frames) + synthetic neonatal dataset (500 patients)*  
*Ethical note: Models trained on adult LUS data + synthetic neonatal data. Clinical validation with real neonatal data required before any clinical use.*
